# EXP09 — Comparação de classificadores e atributos

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Protocolo exploratório

Reproduzimos as doze combinações do EXP-07: quatro classificadores e três grupos de atributos. Serão cinco divisões estratificadas por linha, com semente fixa. **Isto não testa generalização entre contratos** porque há uma identidade positiva única.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from ethmix.execucao import abrir_dados
from ethmix.features import FEATURES

dados = abrir_dados(RAIZ / "data" / "processed" / "features_treino.parquet", ["label", "n_tx", *FEATURES])
atributos = {
    "A": ["n_tx", *FEATURES],
    "B": list(FEATURES),
    "C": ["value_entropy", "fee_entropy", "internal_value_entropy"],
}
seed = 20261119
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
modelos = {
    "Random Forest": RandomForestClassifier(random_state=seed),
    "Árvore": DecisionTreeClassifier(random_state=seed),
    "Gradient Boosting": GradientBoostingClassifier(random_state=seed),
    "Regressão logística": LogisticRegression(max_iter=2000, random_state=seed),
}
linhas = []
for nome, modelo in modelos.items():
    for conjunto, colunas in atributos.items():
        passos = [SimpleImputer(strategy="median")]
        if nome == "Regressão logística":
            passos.append(StandardScaler())
        passos.append(modelo)
        pipeline = make_pipeline(*passos)
        valores = cross_val_score(pipeline, dados[colunas], dados.label.astype(int),
                                   scoring="f1", cv=cv)
        linhas.append({"modelo": nome, "atributos": conjunto,
                       "f1_medio": valores.mean(), "f1_desvio": valores.std()})
resultado = pd.DataFrame(linhas)
display(resultado.pivot(index="modelo", columns="atributos", values="f1_medio").round(4))

## 3. Leitura dos resultados

**Atenção:** os hiperparâmetros do código acima são uma configuração didática da biblioteca e não são apresentados como cópia fiel do EXP-07 original. Para obter paridade numérica com o relatório (RF F1 = 0,9778), devemos recuperar e aplicar exatamente os parâmetros, a limpeza e as divisões arquivadas. Não trate a tabela atual como reprodução dos 12 resultados originais.

In [ ]:
saida = RAIZ / "results" / "generated" / "EXP09"
saida.mkdir(parents=True, exist_ok=True)
resultado.to_csv(saida / "comparacao_exploratoria.csv", index=False)
print("Tabela exploratória salva em", saida)